In [1]:
import os
import json
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

import rasterio
from rasterio.transform import from_origin
import gc
import joblib

In [ ]:
maize = pd.read_csv('../../dataset/region prediction/maize_clustered.csv')
wheat = pd.read_csv('../../dataset/region prediction/wheat_clustered.csv')

In [3]:
maize.columns

Index(['CEC', 'Clay', 'SOC', 'Sand', 'Silt', 'lat', 'lon', 'pH', 'MAT', 'MAP',
       'Elevation', 'Cluster'],
      dtype='object')

In [4]:
wheat.columns

Index(['CEC', 'Clay', 'SOC', 'Sand', 'Silt', 'lat', 'lon', 'pH', 'MAT', 'MAP',
       'Elevation', 'Cluster'],
      dtype='object')

In [3]:
maize_cluster0 = maize[maize["Cluster"] == 0].copy()
wheat_cluster2 = wheat[wheat["Cluster"] == 2].copy()
drop_cols = ['MAT','MAP','Elevation','Cluster']
maize_env = maize_cluster0.drop(columns=drop_cols)
wheat_env = wheat_cluster2.drop(columns=drop_cols)
cb_model = CatBoostRegressor()
cb_model.load_model("../../notebooks/Model/catboost_china_prediction.cbm")
with open("../../notebooks/Model/features_china_prediction.json", "r") as f:
    feature_names = json.load(f)

print("\nTotal features:", len(feature_names))


Total features: 44


In [ ]:
ridge_models = {
    "maize": joblib.load("../../LOSO/maize_fewshot_ridge_20.pkl"),
    "wheat": joblib.load("../../LOSO/wheat_fewshot_ridge_20.pkl")
}

In [5]:
crop_traits = {
    "maize": {
        "Family": "Poaceae",
        "Genus": "Zea",
        "Order": "Poales",
        "monocot": 1,
        "woody": 0,
        "herb": 1,
        "crop": 1,
        "vegetable": 0,
        "legume": 0,
        "grass": 1,
        "perennial": 0,
        "edible": 1,
        "logED": 2.176091259
    },

    "wheat": {
        "Family": "Poaceae",
        "Genus": "Triticum",
        "Order": "Poales",
        "monocot": 1,
        "woody": 0,
        "herb": 1,
        "crop": 1,
        "vegetable": 0,
        "legume": 0,
        "grass": 1,
        "perennial": 0,
        "edible": 1,
        "logED": 2.380211242
    }
}

In [6]:
pfas_table = pd.read_excel("../../dataset/PFAS.xlsx")
pfas_table['logIpc'] = np.log10(pfas_table['Ipc'])
print(pfas_table.shape)
print(pfas_table.head())

(41, 21)
  PFAS_name                                         Raw_SMILES  \
0      PFBA                     C(=O)(C(C(C(F)(F)F)(F)F)(F)F)O   
1     PFPeA              C(=O)(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)O   
2     PFHxA       C(=O)(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)O   
3     PFHpA  C(=O)(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F)(...   
4      PFOA  C(=O)(C(C(C(C(C(C(C(F)(F)F)(F)F)(F)F)(F)F)(F)F...   

   C-F chain length           Ipc  BCUT2D_MRLOW  BCUT2D_MWLOW        SPS  \
0                 3    260.658616     -0.347050     10.147392  14.384615   
1                 4    939.726831     -0.389910     10.044436  15.062500   
2                 5   3314.446802     -0.417659      9.980602  15.526316   
3                 6  11530.539061     -0.436369      9.938399  15.863636   
4                 7  39741.897337     -0.449718      9.909071  16.120000   

   FractionCSP3  VSA_EState7  MaxAbsEStateIndex  ...    Kappa3  HallKierAlpha  \
0      0.750000    -6.602292          11.750579  ...  1.

In [7]:
# ==========================================================
# Remove unnecessary columns
# ==========================================================

drop_cols = ["Raw_SMILES","C-F chain length",'Ipc']

pfas_table = pfas_table.drop(

    columns=drop_cols,

    errors="ignore"
)

In [8]:
# ==========================================================
# Convert descriptor columns to numeric
# ==========================================================

for col in pfas_table.columns:

    if col != "PFAS_name":

        pfas_table[col] = pd.to_numeric(

            pfas_table[col],

            errors="coerce"
        )

In [9]:
# ==========================================================
# Build PFAS_DICT
# ==========================================================

PFAS_DICT = {}

for _, row in pfas_table.iterrows():

    pfas_name = row["PFAS_name"]

    descriptor_dict = row.drop(

        labels=["PFAS_name"]

    ).to_dict()

    PFAS_DICT[pfas_name] = descriptor_dict

In [10]:
# ==========================================================
# Check
# ==========================================================

print("\nTotal PFAS:")

print(len(PFAS_DICT))

print("\nPFAS names:")

print(list(PFAS_DICT.keys())[:10])

print("\nExample:")

example_name = list(PFAS_DICT.keys())[0]

print(example_name)

print(PFAS_DICT[example_name])


Total PFAS:
41

PFAS names:
['PFBA', 'PFPeA', 'PFHxA', 'PFHpA', 'PFOA', 'PFNA', 'PFDA', 'PFUnDA', 'PFDoDA', 'PFTrDA']

Example:
PFBA
{'BCUT2D_MRLOW': -0.347050241178715, 'BCUT2D_MWLOW': 10.1473915373546, 'SPS': 14.3846153846154, 'FractionCSP3': 0.75, 'VSA_EState7': -6.60229166666667, 'MaxAbsEStateIndex': 11.7505787037037, 'qed': 0.712441776552488, 'Kappa3': 1.67669845728037, 'HallKierAlpha': -1.02, 'MinAbsEStateIndex': 3.53256944444444, 'MaxPartialCharge': 0.460094687009651, 'MinAbsPartialCharge': 0.460094687009651, 'MinPartialCharge': -0.476625664235965, 'EState_VSA9': 5.10652739484071, 'MaxAbsPartialCharge': 0.476625664235965, 'PEOE_VSA3': 4.79453718407182, 'logIpc': 2.41607208477418}


In [11]:
maize_SRC = {
    "PFBA":0.031117143,
    "PFPeA":0.024858571,
    "PFHxA":0.016937143,
    "PFHpA":0.013774286,
    "PFOA":0.320571429,
    "PFNA":0.0412,
    "PFDA":0.02014,
    "PFUnDA":0.015994286,
    "PFDoDA":0.002767143,
    "PFTrDA":0.003975714,
    "PFTeDA":0.000463571,
    "PFBS":0.008528571,
    "PFHxS":0.00001,
    "PFOS":0.013094286,
    "PFDS":0.000281429
}

wheat_SRC = {
    "PFBA":0.020134444,
    "PFPeA":0.021268889,
    "PFHxA":0.032223333,
    "PFHpA":0.017078889,
    "PFOA":0.161923333,
    "PFNA":0.131955556,
    "PFDA":0.01247,
    "PFUnDA":0.072088889,
    "PFDoDA":0.002505556,
    "PFTrDA":0.033871111,
    "PFTeDA":0.000603889,
    "PFBS":0.017081111,
    "PFHxS":0.00001,
    "PFOS":0.005995556,
    "PFDS":0.00001
}

In [12]:
# ==========================================================
# Build prediction dataset
# ==========================================================

def build_prediction_dataset(
    env_df,
    crop_name,
    pfas_name,
    tissue
):

    df = env_df.copy()

    # ======================================================
    # Add crop traits
    # ======================================================

    for k, v in crop_traits[crop_name].items():

        df[k] = v

    # ======================================================
    # Add PFAS descriptors
    # ======================================================

    for k, v in PFAS_DICT[pfas_name].items():

        df[k] = v

    # ======================================================
    # Add soil background concentration
    # ======================================================

    if crop_name == "maize":
        df["logSRC"] = np.log10(maize_SRC[pfas_name])
    else:
        df["logSRC"] = np.log10(wheat_SRC[pfas_name])

    # ======================================================
    # Add tissue
    # ======================================================

    df["Tissue"] = tissue

    # ======================================================
    # Fill missing columns
    # ======================================================

    for col in feature_names:

        if col not in df.columns:

            if col in [
                "Family",
                "Genus",
                "Order",
                "Tissue"
            ]:

                df[col] = "Unknown"

            else:

                df[col] = 0

    # ======================================================
    # Reorder
    # ======================================================

    df = df[feature_names]

    return df


In [13]:
def build_calib_features(X, base_pred):

    Xc = pd.DataFrame(index=X.index)

    Xc["base_pred"] = base_pred

    extra_cols = [
        "logSRC",
        "logED",
        "pH",
        "SOC",
        "CEC",
        "Clay",
        "Tissue"
    ]

    for c in extra_cols:
        if c in X.columns:
            Xc[c] = X[c]

    Xc = pd.get_dummies(
        Xc,
        columns=["Tissue"]
    )

    return Xc

In [16]:
maize_env

,CEC,Clay,SOC,Sand,Silt,lat,lon,pH
49782,19.3,35.2,1.99,32.1,32.7,45.118772,85.305550,8.2
49911,21.1,27.9,1.26,30.8,41.3,45.419420,122.935952,7.8
50344,19.6,32.8,3.00,34.4,32.8,45.010015,84.734142,8.2
50346,20.3,35.8,1.68,31.3,32.9,45.124196,85.360940,8.2
50877,19.5,31.1,2.02,35.3,33.6,45.111372,85.314569,8.2
...,...,...,...,...,...,...,...,...
1697131,21.7,24.5,2.75,29.9,45.6,44.595095,82.550977,8.1
1697149,19.6,30.1,2.78,33.3,36.6,45.013846,84.681865,8.1
1697654,20.5,23.4,2.13,29.3,47.3,44.596966,82.581777,8.1
1697680,20.7,36.4,1.92,30.9,32.7,45.123009,85.304031,8.2


In [14]:
ENV_DICT = {
    "maize": maize_env,
    "wheat": wheat_env
}

PREDICT_PFAS = sorted(
    set(maize_SRC.keys()) &
    set(wheat_SRC.keys())
)

CROP_LIST = [
    "maize",
    "wheat"
]

TISSUE_LIST = [
    "root",
    "fruit"
]

In [ ]:
def lnBCF_to_concentration(df, src):

    out = df.copy()

    out["PlantConc"] = np.exp(out["lnBCF_pred"]) * src

    return out

In [21]:
# ==========================================================
# dataframe -> GeoTIFF
# ==========================================================

def dataframe_to_tif(
        df,
        value_col,
        output_tif,
        resolution=0.05,
        nodata=-9999
):

    df = df.copy()

    # ======================================================
    # Quantize coordinates
    # ======================================================

    df["lon"] = (
        np.round(df["lon"] / resolution)
        .astype(np.int32)
        * resolution
    )

    df["lat"] = (
        np.round(df["lat"] / resolution)
        .astype(np.int32)
        * resolution
    )

    # ======================================================
    # Extent
    # ======================================================

    xmin = df["lon"].min()
    xmax = df["lon"].max()

    ymin = df["lat"].min()
    ymax = df["lat"].max()

    # ======================================================
    # Raster size
    # ======================================================

    width = int(
        round((xmax - xmin) / resolution)
    ) + 1

    height = int(
        round((ymax - ymin) / resolution)
    ) + 1

    # ======================================================
    # Empty raster
    # ======================================================

    raster = np.full(
        (height, width),
        nodata,
        dtype=np.float32
    )

    # ======================================================
    # Row / Col
    # ======================================================

    cols = (
        (df["lon"] - xmin)
        / resolution
    ).round().astype(np.int32)

    rows = (
        (ymax - df["lat"])
        / resolution
    ).round().astype(np.int32)

    # ======================================================
    # Fill raster
    # ======================================================

    raster[
        rows,
        cols
    ] = df[value_col].values

    # ======================================================
    # Transform
    # ======================================================

    transform = from_origin(
        xmin - resolution / 2,
        ymax + resolution / 2,
        resolution,
        resolution
    )

    # ======================================================
    # Save
    # ======================================================

    with rasterio.open(
        output_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=np.float32,
        crs="EPSG:4326",
        transform=transform,
        nodata=nodata,
        compress="lzw"
    ) as dst:

        dst.write(raster, 1)

    print(f"GeoTIFF saved: {output_tif}")

In [ ]:
for crop_name in CROP_LIST:

    print("="*80)
    print(crop_name)

    env_df = ENV_DICT[crop_name]

    ridge = ridge_models[crop_name]

    # Ridge训练时的列
    train_columns = ridge.feature_names_in_

    for tissue in TISSUE_LIST:

        print(" Tissue:", tissue)

        for pfas_name in PREDICT_PFAS:

            print("   ", pfas_name)

            #==============================
            # CatBoost input
            #==============================

            X = build_prediction_dataset(
                env_df,
                crop_name,
                pfas_name,
                tissue
            )

            #==============================
            # CatBoost prediction
            #==============================

            base_pred = cb_model.predict(X)

            #==============================
            # Ridge input
            #==============================

            Xc = build_calib_features(
                X,
                base_pred
            )

            # same with training
            Xc = Xc.reindex(
                columns=train_columns,
                fill_value=0
            )

            #==============================
            # Few-shot calibration
            #==============================

            pred = ridge.predict(Xc)

            result = env_df[
                ["lon","lat"]
            ].copy()

            result["lnBCF_pred"] = pred.astype(np.float32)

            # Soil concentration
            if crop_name == "maize":
                src = maize_SRC[pfas_name]
            else:
                src = wheat_SRC[pfas_name]

            # Plant concentration
            result = lnBCF_to_concentration(
                result,
                src
            )

            pkl_file = (
                f"after_calibration/"
                f"{pfas_name}_{crop_name}_{tissue}.pkl"
            )

            csv_file = (
                f"after_calibration/csv/"
                f"{pfas_name}_{crop_name}_{tissue}.csv"
            )

            tif_file = (
                f"after_calibration/tif/"
                f"{pfas_name}_{crop_name}_{tissue}.tif"
            )

            result.to_pickle(pkl_file)

            result.to_csv(
                csv_file,
                index=False,
                float_format="%.6f"
            )

            dataframe_to_tif(
                result,
                value_col="PlantConc",
                output_tif=tif_file,
                resolution=0.005
            )

            print("Saved:", tif_file)

            del X
            del Xc
            del base_pred
            del pred
            del result

            gc.collect()

print("Single PFAS prediction finished.")

maize
 Tissue: root
    PFBA
GeoTIFF saved: after_calibration/tif/PFBA_maize_root.tif
Saved: after_calibration/tif/PFBA_maize_root.tif
    PFBS
GeoTIFF saved: after_calibration/tif/PFBS_maize_root.tif
Saved: after_calibration/tif/PFBS_maize_root.tif
    PFDA
GeoTIFF saved: after_calibration/tif/PFDA_maize_root.tif
Saved: after_calibration/tif/PFDA_maize_root.tif
    PFDS
GeoTIFF saved: after_calibration/tif/PFDS_maize_root.tif
Saved: after_calibration/tif/PFDS_maize_root.tif
    PFDoDA
GeoTIFF saved: after_calibration/tif/PFDoDA_maize_root.tif
Saved: after_calibration/tif/PFDoDA_maize_root.tif
    PFHpA
GeoTIFF saved: after_calibration/tif/PFHpA_maize_root.tif
Saved: after_calibration/tif/PFHpA_maize_root.tif
    PFHxA
GeoTIFF saved: after_calibration/tif/PFHxA_maize_root.tif
Saved: after_calibration/tif/PFHxA_maize_root.tif
    PFHxS
GeoTIFF saved: after_calibration/tif/PFHxS_maize_root.tif
Saved: after_calibration/tif/PFHxS_maize_root.tif
    PFNA
GeoTIFF saved: after_calibration/tif

In [23]:
# # ==========================================================
# # Chain-length grouped lnBCF
# # ==========================================================

CHAIN_GROUPS = {

    "short": [
        "PFBA",
        "PFPeA",
        "PFHxA",
        "PFBS",
        "PFHxS"
    ],

    "medium": [
        "PFHpA",
        "PFOA",
        "PFNA",
        "PFOS"
    ],

    "long": [
        "PFDA",
        "PFUnDA",
        "PFDoDA",
        "PFTrDA",
        "PFTeDA",
        "PFDS"
    ]
}

In [ ]:
for crop_name in CROP_LIST:

    for tissue in TISSUE_LIST:

        for group_name, pfas_list in CHAIN_GROUPS.items():

            print("=" * 60)
            print(group_name, crop_name, tissue)

            conc_stack = []

            coord_df = None

            for pfas_name in pfas_list:

                file = (
                    f"after_calibration/"
                    f"{pfas_name}_{crop_name}_{tissue}.pkl"
                )

                df = pd.read_pickle(file)

                if coord_df is None:

                    coord_df = df[
                        ["lon", "lat"]
                    ].copy()

                conc_stack.append(
                    df["PlantConc"].values
                )

            conc_stack = np.vstack(conc_stack)

            total_conc = np.sum(
                conc_stack,
                axis=0
            ).astype(np.float32)

            result_df = coord_df.copy()

            result_df["PlantConc"] = total_conc

            out_pkl = (
                f"after_calibration/"
                f"{group_name}_{crop_name}_{tissue}.pkl"
            )

            out_csv = (
                f"after_calibration/group_csv/"
                f"{group_name}_{crop_name}_{tissue}.csv"
            )

            out_tif = (
                f"after_calibration/group_tif/"
                f"{group_name}_{crop_name}_{tissue}.tif"
            )

            result_df.to_pickle(out_pkl)

            result_df.to_csv(
                out_csv,
                index=False,
                float_format="%.6f"
            )

            dataframe_to_tif(
                result_df,
                value_col="PlantConc",
                output_tif=out_tif,
                resolution=0.005
            )

            print(result_df["PlantConc"].describe())

            gc.collect()

print("Chain-group prediction finished.")

short maize root
GeoTIFF saved: after_calibration/group_tif/short_maize_root.tif
count    384682.000000
mean          3.147412
std           2.576267
min           0.558676
25%           2.194264
50%           2.642510
75%           3.294748
max         157.020752
Name: PlantConc, dtype: float64
medium maize root
GeoTIFF saved: after_calibration/group_tif/medium_maize_root.tif
count    384682.000000
mean          2.450896
std           2.250829
min           0.499792
25%           1.655932
50%           1.942300
75%           2.568176
max         141.528030
Name: PlantConc, dtype: float64
long maize root
GeoTIFF saved: after_calibration/group_tif/long_maize_root.tif
count    384682.000000
mean          1.810887
std           1.825586
min           0.351123
25%           1.167763
50%           1.349182
75%           1.898858
max         109.484146
Name: PlantConc, dtype: float64
short maize fruit
GeoTIFF saved: after_calibration/group_tif/short_maize_fruit.tif
count    384682.000000
mea